# 🏖️ Holiday Features Manager
- Generate or update `audit_cache.parquet` incrementally
- Inject lists of custom holiday dates
- View holidays for the **rest of 2026** and **all of 2027** from `audit_labels.parquet`
- Launch the **Streamlit · Holiday Audit** app

## 1 · Imports

In [ ]:
import sys
import subprocess
from pathlib import Path
from datetime import date

import pandas as pd

# ── Project root ───────────────────────────────────────────────────────────────
CWD = Path.cwd().resolve()
for candidate in (CWD, CWD.parent, CWD / "analog_holidays", CWD.parent / "analog_holidays"):
    if (
        (candidate / "__init__.py").exists()
        and (candidate / "audit").is_dir()
        and (candidate / "analog").is_dir()
    ):
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the analog_holidays repository from the current working directory.")

PROJ_ROOT = REPO_ROOT.parent
bad_path = str(REPO_ROOT)
while bad_path in sys.path:
    sys.path.remove(bad_path)

loaded_pkg = sys.modules.get("analog_holidays")
if loaded_pkg is not None and not hasattr(loaded_pkg, "__path__"):
    sys.modules.pop("analog_holidays", None)

if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

from analog_holidays.audit.build_cache import main as build_cache_main
from analog_holidays.audit.data_loader import CACHE_PATH, LABELS_PATH, load_audit_df
from analog_holidays.audit.state_manager import save_labels, update_label
from analog_holidays.shared.identify_holidays import load_holidays_catalog

print(f"Project root : {PROJ_ROOT}")
print(f"Cache path   : {CACHE_PATH}")
print(f"Labels path  : {LABELS_PATH}")

## 2 · Notebook parameters
Edit these variables before running the following cells.

In [ ]:
# Series to process
REGIONS_TO_PROCESS = [
    "SEN_demand_SIN",
    "SEN_demand_CEL",
    "SEN_demand_ORI",
    "SEN_demand_OCC",
    "SEN_demand_NOR",
    "SEN_demand_NES",
    "SEN_demand_NTE",
    "SEN_demand_PEN",
]

# Future cache end year
# build_cache.py is hardcoded to 2026-12-31; adjust it there if you need 2027+
FUTURE_CACHE_END_YEAR = 2027

# Additional or custom holiday dates
# Dates that are NOT in holidays_mx.json but should be marked as "holiday".
# Format: "YYYY-MM-DD"
CUSTOM_HOLIDAY_DATES: list[str] = [
    # "2026-08-10",   # example: special bridge day
]

# User name shown in the audit log
AUDIT_USER_ID = "notebook_user"

# Key dates
TODAY        = pd.Timestamp(date.today())
CURRENT_YEAR = TODAY.year
NEXT_YEAR    = CURRENT_YEAR + 1

print(f"TODAY              : {TODAY.date()}")
print(f"Regions            : {len(REGIONS_TO_PROCESS)}")
print(f"Future cache end   : {FUTURE_CACHE_END_YEAR}-12-31")
print(f"Extra holidays     : {len(CUSTOM_HOLIDAY_DATES)}")

## 3 · Build / refresh the cache (incremental)

Calls `build_cache_main()`.  
- If the cache **does not exist** → full build.  
- If the cache **already exists** → only reprocesses regions with new data in the DB; the rest are reused as-is.

In [ ]:
import analog_holidays.audit.build_cache as _bc

# Verify coverage for the configured end year
_bc_text = Path(_bc.__file__).read_text(encoding="utf-8")
if f'"{FUTURE_CACHE_END_YEAR}-12-31"' not in _bc_text:
    print(
        f"⚠  build_cache.py is configured for a different year than {FUTURE_CACHE_END_YEAR}.\n"
        f"   Edit build_cache.py → end_of_year inside main() if you want to extend coverage through {FUTURE_CACHE_END_YEAR}."
    )
else:
    print(f"✅ build_cache.py covers through {FUTURE_CACHE_END_YEAR}-12-31\n")

build_cache_main()

## 4 · Inject custom holidays into `audit_labels.parquet`

Applies `CUSTOM_HOLIDAY_DATES` to every region in `REGIONS_TO_PROCESS`.  
It only has an effect when the list is not empty.

In [ ]:
if not CUSTOM_HOLIDAY_DATES:
    print("ℹ  CUSTOM_HOLIDAY_DATES is empty — cell has no effect.")
else:
    df = load_audit_df()
    changes = 0
    skipped = []
    for raw_date in CUSTOM_HOLIDAY_DATES:
        ts = pd.Timestamp(raw_date)
        for region in REGIONS_TO_PROCESS:
            row = df[(df["unique_id"] == region) & (df["date"] == ts)]
            if row.empty:
                skipped.append((raw_date, region))
                continue
            if row.iloc[0]["label"] != "holiday":
                df = update_label(df, region, ts, "holiday", user_id=AUDIT_USER_ID)
                changes += 1

    if skipped:
        print(f"⚠  {len(skipped)} date/region combinations were not found in the cache:")
        for date_value, region_name in skipped:
            print(f"   {date_value} / {region_name}")

    if changes:
        save_labels(df)
        print(f"\n✅  {changes} labels updated → {LABELS_PATH}")
    else:
        print("ℹ  All dates were already marked as 'holiday' — no changes.")

## 5 · Load all holidays from `audit_labels.parquet`

In [ ]:
df_holidays_all = pd.DataFrame()

if not LABELS_PATH.exists():
    print(
        f"⚠  {LABELS_PATH} does not exist yet.\n"
        "   Use the Streamlit app (cell 8) to audit days and save labels,\n"
        "   or add dates to CUSTOM_HOLIDAY_DATES and run cell 4."
    )
else:
    df_labels_raw = pd.read_parquet(LABELS_PATH)
    df_labels_raw["date"] = pd.to_datetime(df_labels_raw["date"])
    df_holidays_all = (
        df_labels_raw[df_labels_raw["label"] == "holiday"]
        .sort_values(["date", "unique_id"])
        .reset_index(drop=True)
    )
    print(f"Total holiday rows in audit_labels : {len(df_holidays_all):,}")
    print(f"Regions                           : {df_holidays_all['unique_id'].nunique()}")
    print(f"Date range                        : "
          f"{df_holidays_all['date'].min().date()} → {df_holidays_all['date'].max().date()}\n")
    display(df_holidays_all)

## 6 · Holidays for the rest of 2026 (from today)

In [ ]:
if df_holidays_all.empty:
    # Fallback: JSON catalog when audit_labels does not have data yet
    print(f"ℹ  No data in audit_labels — showing holidays from holidays_mx.json for {CURRENT_YEAR}\n")
    df_cat = load_holidays_catalog(PROJ_ROOT / "data" / "holidays_mx.json", CURRENT_YEAR, CURRENT_YEAR)
    df_cat["date"] = pd.to_datetime(df_cat["date"])
    df_rest = df_cat[df_cat["date"] >= TODAY].copy()
    df_rest["date"] = df_rest["date"].dt.strftime("%Y-%m-%d (%A)")
    display(df_rest.rename(columns={"holiday_name": "holiday"}))
else:
    mask = (
        (df_holidays_all["date"] >= TODAY) &
        (df_holidays_all["date"].dt.year == CURRENT_YEAR)
    )
    df_rest = df_holidays_all[mask].copy()
    unique_days = df_rest["date"].nunique()

    # Pivot: one row per date, columns = regions
    df_pivot = (
        df_rest
        .pivot_table(index="date", columns="unique_id", values="label", aggfunc="first")
        .fillna("")
        .reset_index()
    )
    df_pivot["date"] = df_pivot["date"].dt.strftime("%Y-%m-%d (%A)")
    print(f"Pending holidays for {CURRENT_YEAR} ({TODAY.date()} onward): {unique_days} days\n")
    display(df_pivot)

## 7 · Holidays for 2027

In [ ]:
_df_next = (
    df_holidays_all[df_holidays_all["date"].dt.year == NEXT_YEAR].copy()
    if not df_holidays_all.empty
    else pd.DataFrame()
)

if _df_next.empty:
    # Fallback: JSON catalog
    print(f"ℹ  No data for {NEXT_YEAR} in audit_labels — showing holidays from holidays_mx.json\n")
    df_cat_next = load_holidays_catalog(
        PROJ_ROOT / "data" / "holidays_mx.json", NEXT_YEAR, NEXT_YEAR
    )
    df_cat_next["date"] = pd.to_datetime(df_cat_next["date"])
    df_cat_next = df_cat_next.sort_values("date").reset_index(drop=True)
    df_cat_next["date"] = df_cat_next["date"].dt.strftime("%Y-%m-%d (%A)")
    display(df_cat_next.rename(columns={"holiday_name": "holiday"}))
else:
    df_pivot_next = (
        _df_next
        .pivot_table(index="date", columns="unique_id", values="label", aggfunc="first")
        .fillna("")
        .reset_index()
    )
    df_pivot_next["date"] = df_pivot_next["date"].dt.strftime("%Y-%m-%d (%A)")
    print(f"Holidays for {NEXT_YEAR}: {_df_next['date'].nunique()} unique days\n")
    display(df_pivot_next)

## 8 · Launch Streamlit · Holiday Audit

Run the cell to open the app at **http://localhost:8501**.  
The app runs in the background, and the notebook stays active.

In [ ]:
import os
import socket
import time
import webbrowser

STREAMLIT_HOST = "127.0.0.1"
STREAMLIT_PORT = 8501
STREAMLIT_URL = f"http://{STREAMLIT_HOST}:{STREAMLIT_PORT}"
STREAMLIT_LOG_PATH = REPO_ROOT / "audit" / "data" / "streamlit_audit.log"

def _is_port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.2)
        return sock.connect_ex((host, port)) == 0

def _read_log_tail(path: Path, max_chars: int = 4000) -> str:
    if not path.exists():
        return ""
    text = path.read_text(encoding="utf-8", errors="replace")
    return text[-max_chars:].strip()

def _launch_streamlit() -> subprocess.Popen[str]:
    STREAMLIT_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    log_handle = STREAMLIT_LOG_PATH.open("w", encoding="utf-8")
    globals()["_streamlit_log_handle"] = log_handle

    cmd = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(REPO_ROOT / "audit" / "app.py"),
        "--server.headless",
        "true",
        "--server.port",
        str(STREAMLIT_PORT),
        "--browser.gatherUsageStats",
        "false",
    ]

    env = os.environ.copy()
    env["BROWSER"] = "none"

    return subprocess.Popen(
        cmd,
        cwd=str(PROJ_ROOT),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

existing_proc = globals().get("_streamlit_proc")
if existing_proc is not None and existing_proc.poll() is None and _is_port_open(STREAMLIT_HOST, STREAMLIT_PORT):
    print(f"✅  Streamlit · Holiday Audit is already running at {STREAMLIT_URL}")
    webbrowser.open_new_tab(STREAMLIT_URL)
else:
    if existing_proc is not None and existing_proc.poll() is None:
        existing_proc.terminate()

    globals()["_streamlit_proc"] = _launch_streamlit()
    for _ in range(20):
        if _is_port_open(STREAMLIT_HOST, STREAMLIT_PORT):
            print("🚀  Streamlit · Holiday Audit launched")
            print(f"    → {STREAMLIT_URL}")
            print(f"    Log: {STREAMLIT_LOG_PATH}")
            webbrowser.open_new_tab(STREAMLIT_URL)
            break

        if globals()["_streamlit_proc"].poll() is not None:
            log_tail = _read_log_tail(STREAMLIT_LOG_PATH)
            raise RuntimeError(
                "Streamlit exited during startup.\n\n"
                + (log_tail or "No startup log was captured.")
            )

        time.sleep(0.5)
    else:
        log_tail = _read_log_tail(STREAMLIT_LOG_PATH)
        raise RuntimeError(
            "Streamlit did not open port 8501 within 10 seconds.\n\n"
            + (log_tail or "No startup log was captured.")
        )
